# cuTile Python Tiles: Matrix Add

In the previous module you added two vectors. The tiles were one-dimensional: each block handled a contiguous run of elements.

This module extends the same idea to two dimensions using matrix addition. The math is simple on purpose so the focus stays on the tiles.
You will also see how the tile shape you pick affects performance.

## Installing cuTile Python

cuTile Python requires:

    - NVIDIA Kernel Driver R580 or later.
    - CUDA Toolkit 13.1 or later.

You can install cuTile Python via the PIP package `cuda-tile`.

In [ ]:
import os

if not os.getenv("BREV_ENV_ID") and not os.path.exists(os.path.expanduser("~/.accelerated-computing-hub-installed")): # If not running in brev
  print("Installing PIP packages.")
  !pip uninstall "cuda-python" --yes > /dev/null
  !pip install "cuda-tile" "cupy-cuda13x" > /dev/null 2>&1
  open(os.path.expanduser("~/.accelerated-computing-hub-installed"), "a").close()

In [ ]:
import cuda.tile as ct
import cupy as cp
import cupyx as cpx

## Tiles in Two Dimensions

A tile does not have to be one-dimensional. Give `shape` two entries and you get a rectangular tile, and `index` then needs two entries too, one per axis.

A 6x8 array tiled into 3x4 tiles looks like this:

```
            columns 0-3      columns 4-7
          +---------------+---------------+
 rows 0-2 | index=(0, 0)  | index=(0, 1)  |
          +---------------+---------------+
 rows 3-5 | index=(1, 0)  | index=(1, 1)  |
          +---------------+---------------+
```

The rule is the same on each axis independently: tile index `(i, j)` starts at element `(i * 3, j * 4)`.

To give every tile its own block, we launch a **2D grid**. `ct.bid(0)` is the position along the first grid axis and `ct.bid(1)` along the second. The grid tuple you pass to `ct.launch` is always `(x, y, z)`. For a 1D grid you filled in `(n, 1, 1)`; now we fill in the second slot too.

## Example: Matrix Add

Now the kernel. Each block loads one tile from `A`, the matching tile from `B`, adds them, and stores the result in the matching tile of `C`.

Notice how little changes from the vector add version: the shapes and indices gain a second entry. The grid's second slot, which was `1` in module 01, is now `ct.cdiv(a_shape[1], tn)` because tiles now run along columns too. The addition itself is unchanged, because `+` on tiles is elementwise regardless of how many dimensions the tile has.

In [ ]:
@ct.kernel
def matrix_add_tile(A: ct.Array, B: ct.Array, C: ct.Array,
                    tm: ct.Constant[int],   # tile height, in rows
                    tn: ct.Constant[int]):  # tile width, in columns
  row = ct.bid(0)
  col = ct.bid(1)

  a_tile = ct.load(A, index=(row, col), shape=(tm, tn))
  b_tile = ct.load(B, index=(row, col), shape=(tm, tn))

  ct.store(C, index=(row, col), tile=a_tile + b_tile)

Next, let's validate that we've implemented everything correctly.

In [ ]:
a_shape = (4096, 4096)
t_shape = (64, 64)

A = cp.random.uniform(-5, 5, a_shape, dtype=cp.float32)
B = cp.random.uniform(-5, 5, a_shape, dtype=cp.float32)
C = cp.zeros_like(A)

tm, tn = t_shape
grid = (ct.cdiv(a_shape[0], tm), ct.cdiv(a_shape[1], tn), 1)
print(f"{a_shape[0]}x{a_shape[1]} matrix, {tm}x{tn} tiles, grid = {grid}")
print(f"That is {grid[0] * grid[1]:,} tile blocks.")

ct.launch(cp.cuda.get_current_stream(), grid, matrix_add_tile, (A, B, C, tm, tn))

cp.testing.assert_array_almost_equal(C, A + B) # Verify the results
print("\nMatrix add OK")

## Choosing a Tile Shape

A 4096x4096 matrix can be covered by 64x64 tiles, or 128x128 tiles, or 32x256 tiles. All of them are correct. They are not all equally fast.

In two dimensions, size is not the only thing that matters. The shape of a tile affects performance too. Because rows are stored back to back in memory, a wide tile reads more data in one continuous run than a tall tile holding the same number of elements.

Kernel performance depends on both tile size and shape, so let's sweep over a few combinations to see what works best.

In [ ]:
from cuda.core.experimental import Device

# Same helper as module 01.
def get_peak_memory_bandwidth(device_id: int = 0):
  dev = Device(device_id)
  dev.set_current() # Initialize CUDA for this thread

  props = dev.properties
  mem_clock_khz = props.memory_clock_rate        # Peak memory clock in kHz.
  bus_width_bits = props.global_memory_bus_width # DRAM bus width in bits.

  bytes_per_s = (mem_clock_khz * 1_000) * (bus_width_bits / 8) * 2 # DDR factor.
  return bytes_per_s / 2 ** 30

peak_memory_bandwidth = get_peak_memory_bandwidth(0)
if peak_memory_bandwidth == 0:
  print("Note: peak memory bandwidth could not be queried on this device.")
  print("The GB/s column shows real measured throughput.\n")

# Matrix add reads A, reads B, and writes C.
memory_accessed = (3 * A.size * A.dtype.itemsize) / 2 ** 30
print(f"{memory_accessed:.2f} GB moved, {peak_memory_bandwidth:.1f} GB/s peak memory bandwidth\n")

times = []
for tm, tn in [(16, 16), (32, 32), (64, 64), (128, 128),
               (16, 256), (256, 16), (32, 512)]:
  grid = (ct.cdiv(a_shape[0], tm), ct.cdiv(a_shape[1], tn), 1)
  time = cpx.profiler.benchmark(
    lambda: ct.launch(cp.cuda.get_current_stream(), grid, matrix_add_tile, (A, B, C, tm, tn)),
    (), n_repeat=15, n_warmup=5
  ).gpu_times[0].mean()

  times.append({'t_shape': (tm, tn), 'time': time})
  bw = memory_accessed / time
  pct = f"{bw / peak_memory_bandwidth:.2%} of peak" if peak_memory_bandwidth > 0 else "n/a (unified memory)"
  print(f"tile {tm:>3}x{tn:<3}  {time:.3g} s, {bw:.1f} GB/s, {pct}")

from functools import reduce

print()
print(reduce(lambda x, y: x if x['time'] < y['time'] else y, times))

Compare 256x16 against 16x256. They hold the same number of elements but usually do not perform the same. The only difference between them is the shape.

Two things to carry forward: index and shape both gain a second entry when moving from 1D to 2D, and tile shape is a performance knob separate from size.